In [7]:
import spacy
from nltk import word_tokenize
from transformers import AutoTokenizer
import nltk

# для некоторых библиотек требуется ручной вызов загрузки
# необходимых словарей, модулей. 
nltk.download('punkt')

with open("data/tiny_shakespear.txt", "r") as f:
    text = f.read()

# для spacy загрузим отдельную модель/словарь для английского языка en_core_web_sm    
nlp = spacy.load("en_core_web_sm")

# в transformers используем предобученный токенизатор от модели для английского языка
tokenizer = AutoTokenizer.from_pretrained("models/bert-base-cased")

#в nltk указываем язык, с которым работаем явно
tokens_nltk = word_tokenize(text[:60], language='english') 

# для spacy предварительно оборачиваем текст во внутренний формат
# представления документа и используем атрибут .text у токена
doc = nlp(text[:60])
tokens_spacy = [token.text for token in doc]

# токенайзеры transfomers работают с текстами напрямую
tokens_trf = tokenizer.tokenize(text[:60])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Олег\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


OSError: models/bert-base-cased is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [5]:
import spacy

text = "123: hey ds_expert@w.com , check findme.com"

# загрузите модель для английского корпуса из spacy, models/en_core_web_sm
nlp = spacy.load("en_core_web_sm")

# преобразуйте текст во внутренний формат spacy
doc = nlp(text)
tokens = [token.text for token in doc]

# проверьте, является ли токен числом, имейлом, ссылкой
digits = [token.is_digit for token in doc]
emails = [token.like_email for token in doc]
urls = [token.like_url for token in doc]

print("токены:", tokens)
print("числа:", digits)
print("имейлы:", emails)
print("ссылки:", urls)

токены: ['123', ':', 'hey', 'ds_expert@w.com', ',', 'check', 'findme.com']
числа: [True, False, False, False, False, False, False]
имейлы: [False, False, False, True, False, False, False]
ссылки: [False, False, False, False, False, False, True]


In [2]:
import re
from collections import Counter

with open("data/tiny_shakespear.txt", "r") as f:
    text = f.read()
    
pattern = r"(?u)\b\w\w+\b"
tokens_regex = re.findall(pattern, text.lower())

cnt = Counter(tokens_regex)
print(cnt.most_common(5))
print(cnt.most_common()[-5:]) 

[('the', 6287), ('and', 5690), ('to', 4934), ('of', 3760), ('you', 3211)]
[('fowling', 1), ('weakly', 1), ('drowsiness', 1), ('possesses', 1), ('eyelids', 1)]


In [3]:
import pymorphy3

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize


text = "Дети играли в снежки и пришли домой мокрые, но довольные"

stemmer = SnowballStemmer("russian")
tokens = word_tokenize(text, language="russian")

stems = [stemmer.stem(token) for token in tokens]
print("Стемминг (nltk):", stems)

morph = pymorphy3.MorphAnalyzer()
lemms_pm = [morph.parse(token)[0].normal_form for token in tokens]
print("Леммы (pymorphy)", lemms_pm) 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Олег\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Олег\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


Стемминг (nltk): ['дет', 'игра', 'в', 'снежк', 'и', 'пришл', 'дом', 'мокр', ',', 'но', 'довольн']
Леммы (pymorphy) ['ребёнок', 'играть', 'в', 'снежок', 'и', 'прислать', 'домой', 'мокрый', ',', 'но', 'довольный']


In [7]:

import pymorphy3
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords
  

df = pd.read_csv("data/media_articles.csv")
train, test = train_test_split(df, test_size=0.25, stratify=df["category"], random_state=42)
  
# загружаем стоп-слова
stop_words = set(stopwords.words('russian'))

# инициализируем лемматизатор
morph = pymorphy3.MorphAnalyzer()

# функция предобработки: токенизация + лемматизация 
# + нижний регистр + убираем числа
def tokenize_lemmatize(text):
    """убираем стоп-слова и числа, приводим к нижнему регистру, лемматизируем"""
    tokens = word_tokenize(text.lower())
    tokens = [
            word for word in tokens if word.isalpha() and word not in stop_words
        ]
    return [morph.parse(token)[0].normal_form for token in tokens]

# инициализируем и применяем для train датасета TF-IDF
tf_idf = TfidfVectorizer(tokenizer=tokenize_lemmatize,
                        min_df=2,                        
                        max_df=0.95,
                        max_features=10_000)
tf_idf_matrix = tf_idf.fit_transform(train["text"])

# решаем задачу классификации с помощью логистической регрессии
clf = LogisticRegression(C=0.1, random_state=42)
clf.fit(tf_idf_matrix, train["category"])

# визуализируем качество классификации
print(classification_report(test["category"], clf.predict(tf_idf.transform(test["text"]))))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Олег\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Олег\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
d:\Projects\praktikum\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

   athletics       0.94      0.74      0.83       196
   autosport       0.72      0.82      0.77       227
  basketball       0.95      0.57      0.71       184
     extreme       0.51      0.85      0.63       208
    football       0.88      0.70      0.78       209
      hockey       0.74      0.85      0.79       218
   motosport       0.89      0.80      0.84       204
      tennis       0.89      0.95      0.92       220
  volleyball       0.86      0.67      0.75       208
winter_sport       0.81      0.79      0.80       205

    accuracy                           0.78      2079
   macro avg       0.82      0.78      0.78      2079
weighted avg       0.81      0.78      0.78      2079

